# 03 · Quantization: speed, memory and quality

Compare the same model in **FP16**, **8-bit** and **4-bit (NF4)** with `bitsandbytes`, then a
**GGUF Q4_K_M** build with `llama-cpp-python`.

**Model choice:** a 7B model in FP16 needs ~15 GiB for weights alone, so it does not fit in 16 GiB
together with the KV cache. Plan:
- `Qwen2.5-3B-Instruct`: FP16 vs 8-bit vs 4-bit (all fit, clean comparison)
- `Qwen2.5-7B-Instruct`: 8-bit vs 4-bit vs GGUF Q4_K_M

**Install (terminal, inside the venv):**
```bash
pip install bitsandbytes huggingface_hub
# llama-cpp-python with CUDA: prebuilt wheel (pick the cuXXX matching your setup)
pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
# or build from source (needs nvcc):  CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir
```

In [1]:
import os

from fontTools.merge import timer

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc, json, random, re, time, threading
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)
GPU_LABEL = "4060ti-egpu"
DEVICE = "cuda:0"
print(torch.cuda.get_device_name(0))

/home/thai-nguyen/thai/env/ai-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NVIDIA GeForce RTX 4070 Laptop GPU


## 1. A tiny quality set with checkable answers
Free-text answers are hard to grade automatically, so we use 50 arithmetic word problems where the
answer is a single integer. It's a narrow test (math reasoning only), but it is cheap, deterministic
and sensitive enough to show quantization damage. Mention this limitation in your README.

In [2]:
def make_quality_set(n=50, seed=42):
    rnd = random.Random(seed)
    items = []
    for i in range(n):
        a, b, c = rnd.randint(12, 99), rnd.randint(3, 19), rnd.randint(2, 9)
        kind = i % 3
        if kind == 0:
            q = f"A warehouse has {a} boxes. Each truck delivers {b} more boxes, and {c} trucks arrive. How many boxes are there now?"
            ans = a + b * c
        elif kind == 1:
            q = f"A server handles {a} requests per second. After an upgrade it handles {b} times as many, minus {c}. How many requests per second now?"
            a = a * b - c
        else:
            q = f"Tom has {a * c} apples and shares them equally among {c} friends, then buys {b} more for himself. How many apples does Tom have if he kept one share?"
            ans = a + b
        items.append({"id": i, "question": q, "answer": ans})
    return items

QUALITY_SET = make_quality_set()
os.makedirs("../data", exist_ok=True)
with open("../data/quality_set.json", "w") as f:
    for it in QUALITY_SET:
        f.write(json.dumps(it) + "\n")
QUALITY_SET[:2]

[{'id': 0,
  'question': 'A warehouse has 93 boxes. Each truck delivers 6 more boxes, and 2 trucks arrive. How many boxes are there now?',
  'answer': 105},
 {'id': 1,
  'question': 'A server handles 47 requests per second. After an upgrade it handles 10 times as many, minus 5. How many requests per second now?',
  'answer': 105}]

In [3]:
SYSTEM = "Solve the problem. Think briefly, then give the final answer on the last line as: Answer: <number>"

def extract_number(text):
    m = re.search(r"Answer:\s*(-?\d+)", text)
    if m:
        return int(m.group(1))
    nums = re.findall(r"-?\d+", text)
    return int(nums[-1]) if nums else None

## 2. Helpers for transformers models

In [4]:
def load_hf(model_id, mode):
    """mode: 'fp16' | '8bit' | '4bit'"""
    qcfg = None
    if mode == "8bit":
        qcfg = BitsAndBytesConfig(load_in_8bit=True)
    elif mode == "4bit":
        qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_host_memory_stats()
    t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map=DEVICE, quantization_config=qcfg
    ).eval()
    torch.cuda.synchronize()
    return tok, mdl, time.perf_counter() - t0, torch.cuda.memory_allocated() / 1024**3

def chat_inputs(tok, user, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors="pt").to(DEVICE)

def hf_speed(tok, mdl, prompt, n_tokens=256):
    inputs = chat_inputs(tok, prompt)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    out = {}
    def _run():
        with torch.inference_mode():
            out["ids"] = mdl.generate(**inputs, max_new_tokens=n_tokens, min_new_tokens=n_tokens, do_sample=False, streamer=streamer, pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    th = threading.Thread(target=_run); th.start()
    ttft = None
    for chunk in streamer:
        if ttft is None and chunk:
            ttft = time.perf_counter() - t0
    th.join(); torch.cuda.synchronize()
    total = time.perf_counter() - t0
    n = out["ids"].shape[1] - inputs["input_ids"].shape[1]
    return {"ttft_s": ttft, "decode_tok_s": (n - 1) / (total - ttft)}

def hf_quality(tok, mdl, items, max_new_tokens=192):
    correct = 0
    for it in items:
        inputs = chat_inputs(tok, it["question"], SYSTEM)
        with torch.inference_mode():
            ids = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
        text = tok.decode(ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        correct += int(extract_number(text) == it["answer"])
    return correct / len(items)


## 3. Run the experiments
This takes a while (each config answers 50 questions). Watch `nvtop` in another terminal.

In [5]:
PROMPT = "Explain step by step how Kubernetes schedules a pod onto a node."
CONFIGS = [
    ("Qwen/Qwen2.5-3B-Instruct", "fp16"),
    ("Qwen/Qwen2.5-3B-Instruct", "8bit"),
    ("Qwen/Qwen2.5-3B-Instruct", "4bit"),
    ("Qwen/Qwen2.5-7B-Instruct", "8bit"),
    ("Qwen/Qwen2.5-7B-Instruct", "4bit"),
]
rows = []
for model_id, mode in CONFIGS:
    print(f"== {model_id} [{mode}]")
    tok, mdl, load_s, vram = load_hf(model_id, mode)
    for _ in range(2):
        hf_speed(tok, mdl, PROMPT, 32)
    speeds = [hf_speed(tok, mdl, PROMPT, 256) for _ in range(3)]
    acc = hf_quality(tok, mdl, QUALITY_SET)
    rows.append({
        "model": model_id, "quant": mode, "engine": "transformers",
        "load_s": round(load_s, 1),
        "vram_weights_gib": round(vram, 2),
        "vram_peak_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "ttft_ms": round(1000 * sum(s["ttft_s"] for s in speeds) / len(speeds), 1),
        "decode_tok_s": round(sum(s["decode_tok_s"] for s in speeds) / len(speeds), 1),
        "accuracy": acc,
    })
    print(rows[-1])
    del mdl, tok
    gc.collect(); torch.cuda.empty_cache()

== Qwen/Qwen2.5-3B-Instruct [fp16]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:09<00:00, 43.79it/s]


{'model': 'Qwen/Qwen2.5-3B-Instruct', 'quant': 'fp16', 'engine': 'transformers', 'load_s': 13.8, 'vram_weights_gib': 5.75, 'vram_peak_gib': 5.77, 'ttft_ms': 92.2, 'decode_tok_s': 33.9, 'accuracy': 0.66}
== Qwen/Qwen2.5-3B-Instruct [8bit]


Loading weights: 100%|██████████| 434/434 [00:03<00:00, 135.64it/s]


{'model': 'Qwen/Qwen2.5-3B-Instruct', 'quant': '8bit', 'engine': 'transformers', 'load_s': 5.5, 'vram_weights_gib': 3.18, 'vram_peak_gib': 5.77, 'ttft_ms': 174.8, 'decode_tok_s': 19.6, 'accuracy': 0.64}
== Qwen/Qwen2.5-3B-Instruct [4bit]


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 343.65it/s]


{'model': 'Qwen/Qwen2.5-3B-Instruct', 'quant': '4bit', 'engine': 'transformers', 'load_s': 3.2, 'vram_weights_gib': 1.94, 'vram_peak_gib': 5.77, 'ttft_ms': 70.4, 'decode_tok_s': 63.9, 'accuracy': 0.64}
== Qwen/Qwen2.5-7B-Instruct [8bit]


Loading weights:  57%|█████▋    | 192/339 [00:28<00:23,  6.31it/s][W1028 14:16:38.087084459 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 69206016 bytes (free: 18153472, total: 8186822656).
[W1028 14:16:38.096404347 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 69206016 bytes (free: 18153472, total: 8186822656).
Loading weights:  58%|█████▊    | 196/339 [00:29<00:27,  5.15it/s][W1028 14:16:38.764108421 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 136314880 bytes (free: 18153472, total: 8186822656).
[W1028 14:16:38.764245588 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 136314880 bytes (free: 18153472, total: 8186822656).
Loading weights:  58%|█████▊    | 196/339 [00:29<00:21,  6.70it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 17.31 MiB is free. Including non-PyTorch memory, this process has 6.06 GiB memory in use. Of the allocated memory 5.78 GiB is allocated by PyTorch, and 107.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)